<a href="https://colab.research.google.com/github/datsay/kg-enhanced-qa-mintaka/blob/main/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets requests tqdm

### 1. Downloading the Mintaka JSON file

In [2]:
import json
import requests

# 1. Download raw Mintaka test split
url = "https://raw.githubusercontent.com/amazon-science/mintaka/main/data/mintaka_test.json"
print("Downloading Mintaka dataset...")
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
raw_data = response.json()

# 2. Filter for multihop and intersection questions
filtered_data = [
    item for item in raw_data
    if item.get("complexityType") in ["multihop", "intersection"]
][:100]

print(f"Successfully loaded {len(filtered_data)} evaluation samples.")

# 3. Clean and parse entities and gold answers
processed_samples = []
for item in filtered_data:
    # Extract Wikidata entity IDs and labels from questionEntity
    entities = [
        {"id": e.get("name"), "label": e.get("label")}
        for e in item.get("questionEntity", [])
        if e.get("name")
    ]

    # Extract gold answer strings safely
    gold_answers = []
    ans_obj = item.get("answer") or {}

    if ans_obj.get("mention"):
        gold_answers.append(ans_obj["mention"])

    answer_list = ans_obj.get("answer") or []
    for a in answer_list:
        if isinstance(a, dict) and "label" in a:
            label = a["label"]
            if isinstance(label, dict):
                gold_answers.append(label.get("en"))
            elif isinstance(label, str):
                gold_answers.append(label)
        elif isinstance(a, str):
            gold_answers.append(a)

    # Deduplicate answer strings
    gold_answers = list(set([g for g in gold_answers if g]))

    processed_samples.append({
        "id": item["id"],
        "question": item["question"],
        "complexity": item["complexityType"],
        "entities": entities,
        "gold_answers": gold_answers
    })

# 4. Preview the first parsed sample
first = processed_samples[0]
print("\n--- Example Data Sample ---")
print("ID:", first["id"])
print("Question:", first["question"])
print("Complexity:", first["complexity"])
print("Entities:", first["entities"])
print("Gold Answers:", first["gold_answers"])

Successfully loaded 100 evaluation samples.

--- Example Data Sample ---
ID: fae46b21
Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Complexity: intersection
Entities: [{'id': 'Q1497', 'label': 'Mississippi River'}, {'id': 'Q846570', 'label': 'Americans'}]
Gold Answers: ['Mark Twain']


### 2. Knowledge Graph Retrieval (Wikidata SPARQL)

In [3]:
import time

def fetch_wikidata_triples(entity_id, limit=5):
    """
    Fetches up to `limit` direct relational triples for a given Wikidata entity ID.
    Returns triples in human-readable string format: (Subject Label | Predicate Label | Object Label).
    """
    url = "https://query.wikidata.org/sparql"

    # SPARQL query retrieving human-readable labels in English
    sparql_query = f"""
    SELECT ?propLabel ?valLabel WHERE {{
      wd:{entity_id} ?p ?statement .
      ?statement ?ps ?val .
      ?prop wikibase:claim ?p .
      ?prop wikibase:statementProperty ?ps .

      # Filter for items that have an English label
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
      FILTER(isIRI(?val))
    }}
    LIMIT {limit}
    """

    headers = {
        # Custom User-Agent to comply with Wikidata query API policies
        "User-Agent": "AcademicPosterResearchBot/1.0 (nlp-module-poster-project)"
    }

    try:
        response = requests.get(url, params={"query": sparql_query, "format": "json"}, headers=headers, timeout=10)
        if response.status_code == 200:
            results = response.json().get("results", {}).get("bindings", [])
            triples = []
            for r in results:
                prop = r.get("propLabel", {}).get("value")
                val = r.get("valLabel", {}).get("value")
                if prop and val:
                    triples.append(f"{prop} -> {val}")
            return triples
        else:
            return []
    except Exception as e:
        return []

# Test the function on the entities from the first sample
print("Testing Wikidata triple retrieval on the first sample...")
test_sample = processed_samples[0]
print("Question:", test_sample["question"])

for ent in test_sample["entities"]:
    print(f"\nEntity: {ent['label']} ({ent['id']})")
    triples = fetch_wikidata_triples(ent['id'], limit=5)
    for t in triples:
        print(f"  - ({ent['label']}) --[{t}]")
    time.sleep(1) # Polite pause for the API

Testing Wikidata triple retrieval on the first sample...
Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?

Entity: Mississippi River (Q1497)
  - (Mississippi River) --[tributary -> Salt River]
  - (Mississippi River) --[described by source -> Brockhaus and Efron Encyclopedic Dictionary]
  - (Mississippi River) --[tributary -> Des Moines River]
  - (Mississippi River) --[tributary -> Sucker Creek]
  - (Mississippi River) --[tributary -> Cinque Hommes Creek]

Entity: Americans (Q846570)
  - (Americans) --[has part(s) -> Afghan Americans]
  - (Americans) --[has part(s) -> Saudi Americans]
  - (Americans) --[has part(s) -> Tongan Americans]
  - (Americans) --[has part(s) -> Eastern European Americans]
  - (Americans) --[has part(s) -> Bolivian Americans]


### Install huggingface_hub

In [4]:
!pip install -q huggingface_hub

### 3. LLM Inference Function

In [5]:
from google.colab import userdata
from huggingface_hub import InferenceClient

# Retrieve the token
HF_TOKEN = userdata.get('HF_TOKEN')

# Model ID
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(api_key=HF_TOKEN)

def query_llm(prompt, temperature=0.01, max_new_tokens=64):
    """
    Sends a query to Hugging Face Inference Providers with greedy/deterministic decoding.
    """
    messages = [
        {"role": "system", "content": "You are a concise fact-based assistant. Answer the question in as few words as possible."},
        {"role": "user", "content": prompt}
    ]
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=temperature,
            max_tokens=max_new_tokens
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# Test query on the first sample question
test_q = processed_samples[0]["question"]
print("Question:", test_q)
print("Answer:", query_llm(test_q))

Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Answer: Mark Twain.
